In [ ]:
%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

import os
import logging
import yaml
import ipynbname

import pandas as pd

import ferdinand.utils as utl   
import ferdinand.sqlite_utils as sqlutl
import ferdinand.preprocess_and_qc as ppqc

from pathlib import Path
from types import SimpleNamespace


# Purpose and Documentation

## Purpose

This notebook performs automated image-quality checks on the preprocessed radiograph dataset. It provides utilities to flag overexposed images, run quick tests on random samples, compute per-center summaries, and generate a curated overview gallery of images that passed QC. The notebook is designed to be re-runnable and to update the project's SQLite metadata table with QC flags.

## Location 

_&lt;project_dir&gt;/notebooks/320_check_image_quality.ipynb_

## Prerequisites

- A valid runtime configuration file (_../config/config.yml_) with correctly defined _centers_, _run_name_, _parameters_, and _directories_
- Successful execution of the template setup cells from _NNN_notebook_template.ipynb_
- Existing _../data/_ and _../reports/_ directories 
- Existing SQLite database _../data/&lt;run_name&gt;/&lt;all_data_dir&gt;/IMPC_radiographs.db_
- Image files have already been preprocessed and are present in _../data/&lt;run_name&gt;/&lt;preprocessed_dir&gt;_

## Workflow

- Detect and mark images that are overexposed or have other automated quality failures
- Provide quick visual checks (random sampling) and dataset-wide scans 
- Update the SQLite database with QC flags, so downstream steps can filter images

**Overexposure detection is heuristic**: always visually inspect galleries after automated flags before making final exclusion decisions.

# Definition / Settings

This section initializes the notebook environment: 
- loads the runtime configuration from _../config/config.yml_
- creates the required directory structure
- sets up logging and the SQLite connection

**Do not proceed past this section** </br>
Missing paths and an uninitialized database connection will cause subsequent cells to crash.

## General settings

In [ ]:
# load the configuration file
config_path = Path('../config/config.yml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print(pd.DataFrame(config['settings'].items(), columns=['configuration parameter', 'set to']).to_markdown(tablefmt="pretty", index=False))

## Set up runtime paths

In [ ]:
# create directories as per configuration
dirs = SimpleNamespace(**config['directories'])
for key, value in vars(dirs).items():
    dirs.__dict__[key] = os.path.join(config['project_root'], config['data_dir'], config['run_name'], value)
    os.makedirs(dirs.__dict__[key], exist_ok=True)
    if os.path.exists(dirs.__dict__[key]):
        print(f" Directory created: {dirs.__dict__[key]}")
    else:
        print(f" ERROR: Directory does not exist: {dirs.__dict__[key]}")

## Store runtime configuration when necessary

In [ ]:
# copy the current '../config/config.yml' to '../reports/'
dest_filename = f"config_{config['run_name']}.yml"
dest_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], dest_filename)
if not os.path.exists(dest_path): 
    commmand_string = f"cp {config_path} {dest_path}"
    os.system(commmand_string)

    if os.path.exists(dest_path): 
        print(f" Current runtime configuration \'{config_path}' saved to \'{dest_path}\'")
    else:
        print(f" ERROR: could not save configuration file to \'{dest_path}\'")

## Set up logger

In [ ]:
# setup logging
notebook_name = ipynbname.name()

log_path = os.path.join(config['project_root'], config['reports_dir'], config['run_name'], 'logs')
os.makedirs(log_path, exist_ok=True)

log_file = os.path.join(log_path, f"{notebook_name}.log")
print(f"Log file: {log_file}")

try:
    logger = utl.setup_logger(log_file, log_level=logging.INFO)
    print("Logger successfully set up.")
except Exception as e:
    print(f"Error setting up logger: {e}")

## Set up SQLite connection

In [ ]:
# expected SQLlite database file
sqlite_file = os.path.join(dirs.all_data_dir, config['sqlite_db'])

# if already exists, connect to it
if os.path.exists(sqlite_file):
    conn = sqlutl.connect_db(sqlite_file)
    if conn is not None:
        print(f"connected to database {sqlite_file}\n")
else: 
    print(f" ERROR: No SQLite database found!")

# Image quality check

This block performs automated image-quality checks focused on overexposure detection. </br>
What happens here: 
- A small random sample of images is selected and passed to _ppqc.flag_overexposed_images()_ for quick verification and plotting.
- The full set of images (_keep=yes_) is then scanned and flags (_overexposed_, _overall_overexposed_) are written to the database.

Recommended workflow:
1. Run the small random-sample test first and inspect plotted images.
2. If results look correct, run the dataset-wide scan.
3. Review per-center summaries and inspect galleries before deciding on final exclusions.

In [ ]:
my_centers = config['centers']
print(f"Centers to be processed: {my_centers}")

## Flag overexposion on a handful of images

In [ ]:
rnd_rows = sqlutl.select_random_rows_by_column(conn,
                                               db_table='impc_data',
                                               num_rows=4,
                                               column='keep',
                                               value='yes',
                                               logger=logger)

In [ ]:
ppqc.flag_overexposed_images(conn, 
                             db_table='impc_data', 
                             rows=rnd_rows, 
                             source_path=dirs.preprocessed_dir)

rnd_images = []
for idx, row in rnd_rows.iterrows():
    rnd_images.append(os.path.join(utl.build_local_image_dir(dirs.preprocessed_dir, row['center'], row['cohort_type'], row['gene_symbol'], row['sex']), 
                                   f"{row['omero_id']}.jpg"))
    
utl.plot_image_grid(rnd_images, cols=4)

## Flag overexposion center-by-center on all curated images

In [ ]:
rows2keep = sqlutl.select_rows_by_column(conn=conn, 
                                       db_table='impc_data',
                                       column='keep',
                                       value='yes')

total = len(rows2keep)
print(f"Total number of images to preprocess: {total}")

In [ ]:
ppqc.flag_overexposed_images(conn, 
                             db_table='impc_data',
                             rows=rows2keep,
                             source_path=dirs.preprocessed_dir, 
                             logger=logger)

In [ ]:
for col in ['overexposed', 'overall_overexposed']:
    print(f"Number of {col} images")
    for center in my_centers:
        overexposed = len(sqlutl.select_rows_by_columns(conn=conn,
                                                        db_table='impc_data',
                                                        filters={'center': center, col: 'yes', 'keep': 'yes'}))
        logger.info(f"Center: {center}, {col} images: {overexposed}/{total} ({int((overexposed/total)*100) if total > 0 else 0}%)")
        print(f" Center: {center}, {col} images: {overexposed}/{total} ({int((overexposed/total)*100) if total > 0 else 0}%)")
    print()    

# Create thumbnail gallery

Summarizes the curated dataset and creates an interactive HTML gallery of thumbnails that link to the full preprocessed images for quick manual review.

**Requirement**: Preprocessed image thumbnails must be generated beforehand.

In [ ]:
rows = sqlutl.select_rows_by_columns(conn=conn, 
                                     db_table='impc_data', 
                                     filters={'keep': 'yes', 'overexposed': 'no', 'overall_overexposed': 'no'})

In [ ]:
utl.create_thumbnail_gallery(rows, 
                             f"_no_overexposion_dataset.html",
                             f"Clustering-based Curated Dataset with No Overexposion",
                             os.path.join(config['project_root'], config['reports_dir'], config['run_name']),
                             dirs.preprocessed_thumbnails_dir,
                             dirs.preprocessed_dir)